# Exploration

## 1) Laden Sie Ihren Datensatz in das Notebook:

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Pro Welle eine Datei - Trennzeichen, Gross-/Kleinschreibung und Variablennamen unterscheiden sich!
FILES = {2001: "data/HBSC2001OAed1.0_F4.csv",
         2006: "data/HBSC2006OAed1.0_F1.csv",
         2010: "data/HBSC2010OAed1.0_F4.csv",
         2014: "data/HBSC2014OAed1.1_F1.csv",
         2018: "data/HBSC2018OAed1.1.csv"}       # 2018: Semikolon + BOM

raw = {}
for year, path in FILES.items():
    df = pd.read_csv(path, sep=";" if year == 2018 else ",", encoding="utf-8-sig", low_memory=False)
    df.columns = df.columns.str.lower()            # 2014 hat GROSSGESCHRIEBENE Namen
    raw[year] = df

raw[2001].iloc[:3, :12]

,surveyyear,countryno,subregion,schoolno,classno,uniqueid,sampleweights,monhtcollect,yearcollect,sex,grade,monthbirth
0,2002,40000,NaN,400001.0,400001.0,400001001.0,1.0,10,2001,2,1,11.0
1,2002,40000,NaN,400001.0,400001.0,400001002.0,1.0,10,2001,2,1,12.0
2,2002,40000,NaN,400001.0,400001.0,400001003.0,1.0,10,2001,1,1,2.0


## 2) Kurze Beschreibung des Datensatzes: worum geht es und woher haben Sie den Datensatz (Link)?

Schülerbefragung der WHO-Studie *Health Behaviour in School-aged Children* (HBSC) unter 11-, 13- und
15-Jährigen in Europa und Nordamerika. Pro Zeile eine befragte Person mit rund 120–170 Fragen zu Ernährung, Bewegung,
Medienkonsum, Rauchen, **Alkohol**, Cannabis, Wohlbefinden, Schule, ... . Die Studie läuft alle 4 Jahre. Die Folgenden Jahre waren abrufbar: 2001/02, 2006, 2010, 2014 und 2018 (Diese Daten wurden per Email angefragt auf der offiziellen Seite: https://data-browser.hbsc.org/measure/alcohol-consumption-lifetime-use/#chart).

**Für den Vergleich verwendete Spalten** 

| Thema | Variable | Frage | Skala |
|---|---|---|---|
| Sport | `physact60` | An wie vielen der letzten 7 Tage warst du mind. 60 Min. körperlich aktiv? | 0–7 Tage |
| Alkohol | `drunk` (2018: `drunkltm`) | Schon einmal so viel Alkohol, dass du richtig betrunken warst? | 1 = nie … 5 = >10-mal |
| Alkohol (ergänzend) | `alc30d_2` (2014, 2018) bzw. `drink30d` (2010) | Alkohol an wie vielen Tagen in den letzten 30 Tagen? | 1 = nie … 7 = ≥30 Tage |
| Kontext | `sex`, `age`, `agecat`, `countryno` | Geschlecht, Alter, Altersgruppe (11/13/15), Land | – |

...


## 3. Wie groß ist der Datensatz in Bezug auf Zeilen (Anzahl der Elemente) und Spalten (uni-, bi-, multivariat)? Welche Spalten und Zeilen sind relevant für das Projekt?

*Anmerkung: nicht benötigte Spalten können mittels `drop` entfernt werden.*

In [3]:
groesse = pd.DataFrame({y: {"Zeilen (Befragte)": d.shape[0], "Spalten": d.shape[1],
                            "Länder/Regionen": d.countryno.nunique()} for y, d in raw.items()}).T
print(groesse.to_string())
print("\nGesamt:", f"{groesse['Zeilen (Befragte)'].sum():,}".replace(",", "."), "Befragte")

gemeinsam = set.intersection(*[set(d.columns) for d in raw.values()])
print(f"\nIn allen 5 Wellen gleich benannte Spalten: {len(gemeinsam)}")
print(sorted(gemeinsam))

      Zeilen (Befragte)  Spalten  Länder/Regionen
2001             162305      127               35
2006             205938      124               41
2010             213595      128               40
2014             214080      170               41
2018             244097      120               47

Gesamt: 1.040.015 Befragte

In allen 5 Wellen gleich benannte Spalten: 44
['age', 'agecat', 'agesex', 'backache', 'beenbullied', 'bodyheight', 'bodyweight', 'breakfastwd', 'breakfastwe', 'bulliedothers', 'contraceptcondom', 'contraceptpill', 'countryno', 'dizzy', 'fatherhome1', 'feellow', 'fight12m', 'fosterhome1', 'hadsex', 'headache', 'health', 'injured12m', 'irritable', 'lifesat', 'likeschool', 'monthbirth', 'motherhome1', 'nervous', 'physact60', 'schoolpressure', 'sex', 'stepfahome1', 'stepmohome1', 'stomachache', 'studaccept', 'studhelpful', 'studtogether', 'talkfather', 'talkmother', 'talkstepfa', 'talkstepmo', 'thinkbody', 'toothbr', 'yearbirth']


## 4. Gibt es fehlende Werte in dem Datensatz? Wenn ja: Wie gehen Sie damit um?

In [4]:
namen = {"physact60": "physact60", "drunk": "drunk", "drink30d": "drink30d", "alc30d_2": "alc30d_2",
         "alcltm": "alcltm", "alcofreq": "alcofreq", "sampleweights": "sampleweights", "weight": "weight"}
vorhanden = pd.DataFrame({y: {v: (k in d.columns) for k, v in namen.items()} for y, d in raw.items()})
vorhanden.loc["drunk (2018: drunkltm)"] = [("drunk" in d.columns) or ("drunkltm" in d.columns) for d in raw.values()]
print(vorhanden.replace({True: "ja", False: "-"}).to_string())

print("\nDatentypen der Kernvariablen 2018 (alle 'object', weil leere Felder als ' ' gespeichert sind):")
print(raw[2018][["physact60", "drunkltm", "alc30d_2", "agecat", "age"]].dtypes.to_string())
print("\nUngültige Codes 2018 (-99 = 'Missing due to inconsistent answer'):",
      (raw[2018].drunkltm.astype(str).str.strip() == "-99").sum(), "Zeilen bei drunkltm")

                       2001 2006 2010 2014 2018
physact60                ja   ja   ja   ja   ja
drunk                    ja   ja   ja   ja    -
drink30d                  -    -   ja    -    -
alc30d_2                  -    -    -   ja   ja
alcltm                    -    -    -   ja   ja
alcofreq                 ja    -    -    -    -
sampleweights            ja   ja   ja    -    -
weight                    -    -    -    -   ja
drunk (2018: drunkltm)   ja   ja   ja   ja   ja

Datentypen der Kernvariablen 2018 (alle 'object', weil leere Felder als ' ' gespeichert sind):
physact60    object
drunkltm     object
alc30d_2     object
agecat       object
age          object

Ungültige Codes 2018 (-99 = 'Missing due to inconsistent answer'): 697 Zeilen bei drunkltm


**Befund:** Trennzeichen (`,` vs. `;`), Zeichenkodierung (BOM in 2018), Groß-/Kleinschreibung (2014) und Variablennamen
(`drunk` vs. `drunkltm`, `drink30d` vs. `alc30d_2`) unterscheiden sich; 2018 enthält leere Felder als Leerzeichen und den
Sondercode `-99`. Ohne Bereinigung würden 2018-Werte als Text eingelesen; das Alter (Dezimalkomma, z. B. 13,5) ginge beim Umwandeln zu 91 % verloren.

### Task Abstraction

5. Welche Fragen können mit dem gewählten Datensatz untersucht werden?

*Anmerkung: Diese Fragen können in einem domänenspezifischen Format gestellt werden. Hinterfragen Sie hier auch: welche Fragen benötigen eine visuelle Analyse? Nicht nur Bestimmung von Anzahl und Min/Max*

...

6. Welche abstrakten Aufgaben stecken in den Fragen? *(z.B. Finden von Ausreißern, Vergleich von mehreren Werten)*

...

### Datentransformation

7. Inwiefern ist es notwendig den Datensatz für die Beantwortung der Fragen anzupassen (z.B. Verknüpfungen, Filterung, Änderung des Detailgrads)? Führen Sie diese Datentransformationen in Python durch und binden den finalen Datensatz nochmal tabellarisch ein, da sich die weiteren Aufgaben darauf beziehen.

---

### Data Abstraction

8. Welche Attributtypen können den einzelnen Spalten zugewiesen werden? Geben Sie für ordinale und nominale Daten an, wieviele eindeutige Klassen pro Spalte enthalten sind! Geben Sie für quantitative Daten den Wertebereich an (Minimum und Maximum) (in Python mit min/max bzw. unique())!

9. Stecken weitere semantische Datenstrukturen in dem Datensatz (z.B. hierarchisch, geografisch, temporal), die für die Visualisierungen nützlich sein können?

...

---

### Visuelle Exploration des Datensatzes

10. Explorieren Sie die Eigenschaften des Datensatzes mit mindestens zwei interaktiven (verschiedene) Visualisierungen pro Teammitglied! (z.B. Verteilungen oder Korrelationen untersuchen)

11. Was haben Sie über die Eigenschaften der Daten erfahren? Welche Muster haben Sie erkannt? Welche Insights sind interessant für die Explanation Phase? (dies kann auch direkt unter jeder Visualisierung beantwortet werden)

...